# 2. Predicting eCLIP signal

This notebook runs the model on real sequence and reads what comes out: three tracks per
task, each a density over positions.

The point to internalise is that PARNET predicts **shape, not depth**. Every track is a
probability distribution over the window, so it tells you where signal falls, not how
much of it there is. Comparing a prediction to data therefore means comparing shapes, and
the right statistic is a correlation over positions rather than an error in read counts.

In [1]:
import torch

torch.set_num_threads(2)

import numpy as np
import parnet_tutorials as pt
from parnet.models import RBPNet
from parnet.tokenization import RnaTokenizer

MODEL = 'parnet-21m-penalty5'  # change this one line to run the notebook on another model
model = RBPNet.from_pretrained(pt.model_path(MODEL))
task_names = pt.model_tasks(MODEL)
print('tasks:', len(task_names))

tasks: 223


/home/user/projects/parnet-project/parnet--analyses--interpretation/.pixi/envs/parnet-dev-cu12/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


## From sequence to prediction

The tokenizer accepts DNA or RNA, upper or lower case, and is explicit about everything
else. This matters more than it sounds: the old helper silently turned a lowercase
sequence into an all-zero tensor and the model still returned a confident prediction.
Soft-masked genome sequence is lowercase, so that bug was reachable from ordinary input.

In [2]:
tokenizer = RnaTokenizer()

for seq in ['ACGT' * 4, 'acgt' * 4, 'ACGU' * 4]:
    encoded = tokenizer(seq)
    ids = encoded['input_ids']
    print(f'{seq[:8]:10s} -> ids {ids[0, :8].tolist()}  (sum {int(ids.sum())})')

print()
print('DNA and RNA and case all give the same ids:',
      torch.equal(tokenizer('ACGT')['input_ids'], tokenizer('acgu')['input_ids']))

ACGTACGT   -> ids [0, 1, 2, 3, 0, 1, 2, 3]  (sum 24)
acgtacgt   -> ids [0, 1, 2, 3, 0, 1, 2, 3]  (sum 24)
ACGUACGU   -> ids [0, 1, 2, 3, 0, 1, 2, 3]  (sum 24)

DNA and RNA and case all give the same ids: True


/data/projects/lambosaur/parnet-refactor/parnet/tokenization.py:138: UserWarning: Lowercase bases were upper-cased. In genomic FASTA lowercase marks soft-masked repeats, which is real sequence; it is not dropped. Pass warn_on_lowercase=False to silence this.
  ids = [self.encode(s) for s in sequences]


In [3]:
rng = np.random.default_rng(0)
sequence = ''.join(rng.choice(list('ACGT'), size=600))

with torch.no_grad():
    out = model(**tokenizer(sequence))

for key, value in out.items():
    print(f'{key:12s} {tuple(value.shape)}')

target       (1, 223, 600)
control      (1, 223, 600)
total        (1, 223, 600)
mix_coeff    (1, 223)
target_weight (1, 223)


`target`, `control` and `total` are **log-probabilities** over the window positions, in
natural log, for each task. They are not logits in the pre-sigmoid sense: exponentiating a
profile and summing along the length axis gives 1.

That is true of `target` and `control` for every model. It is **not** true of `total` for
the 21M, and the reason is the mixing rule, so it is worth seeing rather than being told.

In [4]:
# logsumexp along the length axis is exactly 0 for a normalised log-probability profile.
# We check it rather than exponentiating, because exp() of an unnormalised profile
# overflows float32 and prints `inf`, which tells you nothing about how far off it is.
for key in ('target', 'control', 'total'):
    lse = out[key].double().logsumexp(-1)
    off = lse.abs()
    print(f'{key:7s} |logsumexp| over length:  max {float(off.max()):9.3g}   '
          f'not normalised: {int((off > 1e-3).sum())} of {off.numel()} (sequence, task) pairs')

print()
print('target and control are normalised. For the 21M, total is not.')

target  |logsumexp| over length:  max  4.46e-07   not normalised: 0 of 223 (sequence, task) pairs
control |logsumexp| over length:  max  4.79e-07   not normalised: 0 of 223 (sequence, task) pairs
total   |logsumexp| over length:  max       712   not normalised: 173 of 223 (sequence, task) pairs

target and control are normalised. For the 21M, total is not.


## Why `total` is not a distribution for the 21M

The 21M combines the two tracks in log space, adding the mixing coefficient to the target
track. That coefficient is an unbounded logit, so the sum shifts the whole profile by a
constant: the **shape** is right, the offset is not a probability. The 3M backbones use the
convex rule instead, which is a weighted average of two normalised profiles, so their
`total` stays normalised.

This matters whenever you treat a profile as a distribution — plotting relative heights is
fine either way, but anything that integrates probability mass is not. Normalise first:

```python
probs = out['total'].softmax(-1)
```

and note what the raw numbers look like, so an unnormalised profile is recognisable:

In [5]:
profile = out['total'][0, 0]
print(f'raw total     : min {float(profile.min()):8.2f}  max {float(profile.max()):8.2f}  '
      f'sum {float(profile.sum()):10.1f}')

import math
length = profile.shape[-1]
print(f'uniform at L={length} would sum to {length * math.log(1 / length):.1f}')
print('A normalised profile sums to AT MOST that, and less the peakier it is,')
print('so a sum well above it means the profile carries an offset.')
print()

normalised = out['total'][0, 0].softmax(-1)
print(f'after softmax : sums to {float(normalised.sum()):.6f}, '
      f'peak {float(normalised.max()):.5f} at position {int(normalised.argmax())}')

raw total     : min   -10.37  max    -3.67  sum    -4107.2
uniform at L=600 would sum to -3838.2
A normalised profile sums to AT MOST that, and less the peakier it is,
so a sum well above it means the profile carries an offset.

after softmax : sums to 1.000000, peak 0.02550 at position 298


## Reading one task

Pick a task by name rather than by index. Indices are a property of the dataset the model
was trained on and are easy to get wrong.

In [6]:
# Which tracks exist depends on the model: the 21M emits 223 tracks across both cell
# types, each 3M backbone only its split's training RBPs, all HepG2. Ask for a role.
rbp = pt.demo_track(MODEL)
idx = pt.track_index(MODEL, rbp)
print(f'{rbp} is task index {idx}')

target = out['target'][0, idx].exp().numpy()
control = out['control'][0, idx].exp().numpy()
total = out['total'][0, idx].exp().numpy()

print(f'target  peak at position {target.argmax():3d}, height {target.max():.5f}')
print(f'control peak at position {control.argmax():3d}, height {control.max():.5f}')
print(f'total   peak at position {total.argmax():3d}, height {total.max():.5f}')

w = torch.sigmoid(out['mix_coeff'][0, idx]).item()
print(f'\ntarget weight w for this task: {w:.4f}')
print('total is dominated by', 'target' if w > 0.5 else 'control')

HNRNPC_HepG2 is task index 77
target  peak at position  55, height 0.04472
control peak at position 378, height 0.01570
total   peak at position  55, height 0.20827

target weight w for this task: 0.8161
total is dominated by target


## Plot it

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(target, label='target', linewidth=1.0)
ax.plot(control, label='control', linewidth=1.0)
ax.plot(total, label='total', linewidth=1.4, color='black')
ax.set_xlabel('position in window (nt)')
ax.set_ylabel('predicted density')
ax.set_title(f'{rbp}, random sequence, w = {w:.3f}')
ax.legend(frameon=False)
fig.tight_layout()

# Figures go to results/, which is gitignored. Nothing a notebook produces belongs in
# the repository: it is regenerated by running the notebook.
results = pt.project_root() / 'results'
results.mkdir(exist_ok=True)
fig.savefig(results / '02_profile.png', dpi=120)
print('saved', results / '02_profile.png')

saved /data/projects/lambosaur/parnet-tutorials/results/02_profile.png


## Batching

Windows are independent, so batching is free parallelism and the model is fully
convolutional, so a batch is one call.

Two things worth checking rather than assuming: that the answer does not depend on batch
size, and that the output shapes do not either. The second was not true until recently:
`SequenceLinearMix` squeezed every size-1 axis, so at batch size 1 the batch axis
vanished and `mix_coeff` came back with shape `(tasks,)` instead of `(1, tasks)`.

In [8]:
import time

sequences = [''.join(rng.choice(list('ACGT'), size=600)) for _ in range(16)]
encoded = tokenizer(sequences)
print('batched ids:', tuple(encoded['input_ids'].shape))

started = time.time()
with torch.no_grad():
    batched = model(**encoded)
elapsed = time.time() - started
print(f'{len(sequences)} windows in {elapsed:.2f} s on CPU '
      f'({elapsed / len(sequences) * 1000:.0f} ms per window)')
print('output:', tuple(batched['total'].shape))

# Does batching change the answer? Only by float32 round-off: convolutions pick
# different kernels at different batch sizes. Worth measuring rather than assuming.
with torch.no_grad():
    alone = model(**tokenizer(sequences[0]))

difference = (alone['total'] - batched['total'][:1]).abs()
scale = batched['total'][:1].abs().max()
print()
print(f'alone vs in-batch: max abs diff {difference.max():.2e} on values up to {scale:.1f}')
print(f'relative: {difference.max() / scale:.1e}')

profile_alone = alone['total'][0, 0].exp().numpy()
profile_batch = batched['total'][0, 0].exp().numpy()
print(f'profile correlation: {np.corrcoef(profile_alone, profile_batch)[0, 1]:.10f}')
print()
print('Shapes do not depend on batch size either:')
print('  batch of 1 :', tuple(alone['mix_coeff'].shape))
print('  batch of 16:', tuple(batched['mix_coeff'].shape))

batched ids: (16, 600)


16 windows in 1.65 s on CPU (103 ms per window)
output: (16, 223, 600)

alone vs in-batch: max abs diff 1.22e-04 on values up to 676.1
relative: 1.8e-07
profile correlation: 1.0000000000

Shapes do not depend on batch size either:
  batch of 1 : (1, 223)
  batch of 16: (16, 223)


## Comparing to data: use a correlation, not an error

Because predictions are shapes, the per-window statistic that means something is the
Pearson correlation between the predicted density and the observed read profile over
positions. Windows with no reads carry no shape at all and must be dropped before
averaging, or they dilute the result with noise.

The helper below is the whole evaluation idea in eight lines.

In [9]:
def profile_pcc(predicted, observed, min_reads=10):
    """Pearson correlation over positions, for windows with enough signal to have a shape.

    Returns NaN for windows below min_reads rather than a misleading number. Averaging
    over NaN-free entries only is the caller's job, and is deliberate: silently treating
    an empty window as correlation 0 is how an evaluation gets quietly pessimistic.
    """
    observed = np.asarray(observed, dtype=float)
    if observed.sum() < min_reads:
        return float('nan')
    predicted = np.asarray(predicted, dtype=float)
    if predicted.std() == 0 or observed.std() == 0:
        return float('nan')
    return float(np.corrcoef(predicted, observed)[0, 1])


# Demonstrate on synthetic observations: a window that follows the prediction, one that
# does not, and one with almost no reads.
follows = rng.poisson(target * 5000)
unrelated = rng.poisson(np.roll(target, 300) * 5000)
empty = rng.poisson(target * 0.5)

for label, observed in [('follows prediction', follows),
                        ('shifted by 300 nt', unrelated),
                        ('almost no reads', empty)]:
    value = profile_pcc(total, observed)
    print(f'{label:20s} reads {observed.sum():7.0f}  PCC {value:.3f}' if value == value
          else f'{label:20s} reads {observed.sum():7.0f}  dropped (below threshold)')

follows prediction   reads    5012  PCC 0.979
shifted by 300 nt    reads    5014  PCC -0.002
almost no reads      reads       0  dropped (below threshold)


## What to take away

1. Predictions are densities over positions. There is no depth in them.
2. Address tasks by name; indices belong to the dataset.
3. The tokenizer handles case and DNA/RNA explicitly, so soft-masked sequence is safe.
4. Evaluate with a per-window correlation, and drop windows with too few reads rather
   than scoring them as zero.